# Differentiation Class

This notebook prototypes a class to use PyNumero for symbolic and automatic differentiation

In [5]:
import pyomo.environ as pyo

from pyomo.core.expr.calculus.diff_with_pyomo import reverse_sd, reverse_ad
from pyomo.core.expr.visitor import identify_variables
from pyomo.common.collections import ComponentSet

import numpy as np

import pandas as pd

from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment # Load the example

# For IPOPT executable
# import idaes

# Required if reading in files.
import json

In [6]:
class ExperimentGradients:
    def __init__(self, experiment_model, symbolic=True, automatic=True, verbose=False):
        """
        Initialize the ExperimentGradients class.
        Parameters
        ----------
        experiment_model : Pyomo model
            The experiment model to analyze.
        symbolic : bool, optional
            If True, perform symbolic differentiation. Default is True.
        automatic : bool, optional
            If True, perform automatic differentiation. Default is True.
        
        Performance tip:
        - If you are only interested in one type of differentiation (symbolic or automatic),
        you can set the other to False to save computation time.
        - If you will use the instance of this class to perform both symbolic and automatic differentiation,
        you can set both symbolic and automatic to True here.
        - This implementation assumes the experiment model will not be modified after this class is initialized.

        """
        
        self.model = experiment_model

        self.verbose = verbose

        self._analyze_experiment_model()

        self.jac_dict_sd = None
        self.jac_dict_ad = None
        self.jac_measurements_wrt_param = None
        
        if symbolic or automatic:
            # Analyze the experiment model to get the constraints and variables
            self._perform_differentiation(symbolic, automatic)

    def _analyze_experiment_model(self):
        """
        Partition the experiment model constraints and variables into
        sets for equality constraints, outputs (measurements), inputs,
        unknown parameters.

        This will build list of indices used for the performaning
        symbolic differentiation and automatic differentiation.
        """

        model = self.model

        # Parameters
        # Create an empty component set
        param_set = ComponentSet()

        # Loop over the unknown model parameters
        for p in model.unknown_parameters.keys():
            param_set.add(p)

        # Assemble into a list
        
        param_list = list(param_set)

        # Measurements (outputs)
        # Create an empty component set
        output_set = ComponentSet()

        # Loop over the model outputs
        for o in model.experiment_outputs.keys():
            output_set.add(o)

        # Assemble into a list
        output_list = list(output_set)

        # Constraints and Variables
        # Create empty component sets
        con_set = ComponentSet() # These will be all constraints in the Pyomo model
        var_set = ComponentSet() # These will be all Pyomo variables in the Pyomo model

        # Loop over the active model constraints
        for c in model.component_data_objects(pyo.Constraint, descend_into=True, active=True):

            # Add constraint c to the constraint set
            con_set.add(c)

            # Loop over the variables in the constraint c
            # Note: changed this to include_fixed=True
            # Changed back to False to fix problem degree of freedom issues
            for v in identify_variables(c.body, include_fixed=False):
                # Add variable v to the variable set
                var_set.add(v)

        # recall that the parameters are fixed, so we did not
        # get them above. Let's add them now.
        for p in model.unknown_parameters.keys():
            var_set.add(p)

        # Assemble into lists
        con_list = list(con_set)
        var_list = list(var_set)

        # Create empty lists
        param_index = []
        model_var_index = []
        measurement_index = []
        # Adding a `included` suffix to only
        # take outputs that are unfixed. This
        # makes indices match.
        measurement_error_included = pyo.Suffix(direction=pyo.Suffix.LOCAL)

        # Loop over the variables and determine which ones 
        # (and associated indices) are (a) parameters or 
        # (b) measurements
        # TODO: Does this considered fixed variables?
        # How does that change things? We fix all of our
        # experiment inputs and unknown parameters.
        for i, v in enumerate(var_set):
            # Check if the variable is a parameter
            if v in param_set:
                # If yes, record its index
                param_index.append(i)
            else:
                # Otherwise, it is a model variable
                model_var_index.append(i)

                # Check if the model variable is a measurement
                if v in output_set:
                    # If yes, record its index
                    measurement_index.append(i)
                    measurement_error_included[v] = model.measurement_error[v]

        # TODO: Check lengths here. The experiment model should be square if
        # the experiment inputs and unknown parameters are fixed.

        num_measurements = len(output_set)
        num_params = len(param_set)
        num_constraints = len(con_set)
        num_vars = len(var_set)
        num_inputs = len(model.experiment_inputs)

        if self.verbose:
            print("Experiment model size:")

            print(f"  {num_vars} total variables")
            print(f"  {num_measurements} outputs (measurements)")
            print(f"  {num_inputs} inputs")
            print(f"  {num_params} unknown parameters")
            print(f"  {num_constraints} constraints\n")

        if num_vars - num_params != num_constraints:
            raise ValueError("The model is not square: the number of variables minus unknown parameters does not equal the number of constraints.\n" \
            "This is required for the (automatic) differentiation to work correctly.")

        # Save terms that are needed for later
        self.con_list = con_list
        self.var_list = var_list
        self.param_list = param_list

        self.param_index = param_index
        self.model_var_index = model_var_index
        self.measurement_index = measurement_index
        
        self.measurement_error_included = measurement_error_included
        
        self.num_measurements = num_measurements
        self.num_params = num_params
        self.num_constraints = num_constraints
        self.num_vars = num_vars

    def _perform_differentiation(self, symbolic=True, automatic=True):
    
        # Initialize dictionaries to hold the Jacobian entries
        if symbolic:
            jac_dict_sd = {}
        if automatic:
            jac_dict_ad = {}

        if not symbolic and not automatic:
            raise ValueError("At least one differentiation method must be selected: symbolic or automatic.")

        # Grab data needed for the differentiation
        con_list = self.con_list
        var_list = self.var_list

        # Enumerate over the constraints
        for i,c in enumerate(con_list):
            # Check we only have equality constraints... otherwise this gets more complicated
            assert c.equality, "This function only works with equality constraints"
            
            # Perform symbolic differentiation
            if symbolic:
                der_map_sd = reverse_sd(c.body)
            
            if automatic:
                der_map_ad = reverse_ad(c.body)

            # Loop over the Pyomo variables, which includes 
            # parameters, measurements, control decisions
            for j,v in enumerate(var_list):

                # Symbolic differentiation
                if symbolic:
                    # Check if the variable is in the derivative map
                    if v in der_map_sd:
                        # Record the expression 
                        deriv = der_map_sd[v]
                    else:
                        # Otherwise, record 0
                        deriv = 0
                    # Save results in the Jacobian dictionary
                    jac_dict_sd[(i, j)] = deriv

                # Automatic differentiation
                if automatic:
                    if v in der_map_ad:
                        # Record the expression 
                        deriv = der_map_ad[v]
                    else:
                        # Otherwise, record 0
                        deriv = 0
                    # Save results in the Jacobian dictionary
                    jac_dict_ad[(i, j)] = deriv

        if symbolic:
            self.jac_dict_sd = jac_dict_sd
        if automatic:
            self.jac_dict_ad = jac_dict_ad

    def compute_gradient_outputs_wrt_unknown_parameters(self):
        """ Perform automatic differentiation to compute the gradients of the outputs 
        with respect to the unknown parameters.
        
    
        """

        if self.jac_dict_ad is None:
            # Perform automatic differentiation if not already done
            self._perform_differentiation(symbolic=False, automatic=True)

        # Grab the necessary data from the instance
        # (this keeps variable names shorter below)
        num_constraints = self.num_constraints
        num_params = self.num_params
        param_index = self.param_index
        model_var_index = self.model_var_index
        jac_dict_ad = self.jac_dict_ad
        measurement_index = self.measurement_index
    
        jac_con_wrt_param = np.zeros((num_constraints, num_params))
        for i in range(num_constraints):
            for j, p in enumerate(param_index):
                jac_con_wrt_param[i, j] = jac_dict_ad[(i, p)]

        jac_con_wrt_vars = np.zeros((num_constraints, len(model_var_index)))
        for i in range(num_constraints):
            for j, v in enumerate(model_var_index):
                jac_con_wrt_vars[i, j] = jac_dict_ad[(i, v)]

        if self.verbose:
            print(f"Jacobian of constraints with respect to parameters shape: {jac_con_wrt_param.shape}")
            print(f"Jacobian of constraints with respect to variables shape: {jac_con_wrt_vars.shape}")

        jac_vars_wrt_param = np.linalg.solve(
            jac_con_wrt_vars, -jac_con_wrt_param
        )

        # print(f"Jacobian of all variables with respect to parameters:\n{jac_vars_wrt_param}")

        jac_measurements_wrt_param = jac_vars_wrt_param[measurement_index, :]

        # print(f"Jacobian of measurements with respect to parameters:\n{jac_measurements_wrt_param}")

        self.jac_measurements_wrt_param = jac_measurements_wrt_param

        return jac_measurements_wrt_param
    
    def _package_jac_as_df(self, jac):
        """
        Convert a numpy array containing the Jacobian into a
        pandas DataFrame

        Arguments:
            jac: numpy array where rows are measurements and columns are parameters

        Returns:
            pandas DataFrame

        """

        var_list = self.var_list
        measurement_index = self.measurement_index
        param_index = self.param_index

        row_names = [str(var_list[y]) for y in measurement_index]
        col_names = [str(var_list[p]) for p in param_index]

        return pd.DataFrame(jac, index=row_names, columns=col_names)

    def get_numeric_sensitivity_as_df(self):
        if not self.jac_measurements_wrt_param:
            self.compute_gradient_outputs_wrt_unknown_parameters()

        return self._package_jac_as_df(self.jac_measurements_wrt_param)


    def construct_sensitivity_constraints(self, block):

        if self.jac_dict_sd is None:
            # Perform symbolic differentiation if not already done
            self._perform_differentiation(symbolic=True, automatic=False)


In [7]:
from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment # Load the example

data_ex = {"CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0.0, 1.0], "control_points": {0: 500, 0.125: 450, 0.25: 400, 0.375: 350, 0.5: 300, 0.625: 300, 0.75: 300, 0.875: 300, 1: 300}, "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05}

experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)

# Create a Pyomo model
model = experiment.get_labeled_model()

# Loop over the design variables, fix them
for v in model.experiment_inputs:
    v.fix()

print("Solving the square model with design variables fixed...")
# Solve the model
solver = pyo.SolverFactory('ipopt')
results1 = solver.solve(model, tee=True)

exp_grads = ExperimentGradients(model, symbolic=False, automatic=True, verbose=True)

df = exp_grads.get_numeric_sensitivity_as_df()

print(df)


Solving the square model with design variables fixed...
Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for